In [ ]:
import mapof.marriages as mapof
import pandas
import ast
from ast import literal_eval
import numpy as np
from printing import print_map_2d_colored_by_feature
import json

In [9]:
from ast import literal_eval
import pandas as pd

def parse_soc_file(filepath):
    """Parse a .soc file into a list of ballots (one per voter)."""
    ballots = []
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    data_lines = [l.strip() for l in lines if not l.startswith('#') and l.strip()]
    for line in data_lines:
        count, ranking = line.split(':')
        count = int(count.strip())
        ranking = [int(x) for x in ranking.strip().split(',')]
        ballots.extend([ranking] * count)
    
    return ballots

In [139]:
import random
ballots_men = []
ballots_women = []
for i in range(0,150):
    men = parse_soc_file(f'00004_netflix/00004-00000{random.randint(101,200)}.soc')
    women = parse_soc_file(f'00004_netflix/00004-00000{random.randint(101,200)}.soc')
    chosen_ballots_men = []
    chosen_ballots_women = []
    for z in range(4):
        chosen_ballots_men.append(men[random.randint(0, len(men)-1)])
        chosen_ballots_women.append(women[random.randint(0, len(women)-1)])
    chosen_ballots_men = str(chosen_ballots_men)
    chosen_ballots_women = str(chosen_ballots_women)
    ballots_men.append(chosen_ballots_men)
    ballots_women.append(chosen_ballots_women)


In [140]:
df = pd.DataFrame({
    "Mpref": ballots_men,
    "Wpref": ballots_women
})

In [141]:
pd.DataFrame.to_csv(df, 'netflix_preferences.csv', index=False)

**Experiments with Netflix Data**

In [165]:
#Run Gale-Shapley

def gale_shapley(mpref, wpref, strategic_woman=0):
    n = len(mpref)
    curr_match_count = 0
    last_rejected_by_rank_what = [0 for i in range(n)]
    men_matchings = [0 for i in range(n)]
    women_matchings = [0 for i in range(n)]
    proposals_to_strategic_woman = set()    # Only used if strategic_woman != 0

    # continue until all matches made
    while curr_match_count < n:
        for man in range(n):
            # if man is single
            if men_matchings[man] == 0:
                # get first non-rejecting woman from man's preference list
                woman = mpref[man][last_rejected_by_rank_what[man]] - 1
                if woman + 1 == strategic_woman:
                    proposals_to_strategic_woman.add(man + 1)

                # both man and woman are single
                if women_matchings[woman] == 0:
                    # create match (man, woman)
                    men_matchings[man] = woman + 1
                    women_matchings[woman] = man + 1
                    curr_match_count += 1

                # woman is with someone
                else:
                    # find ranks of curr_partner and man given woman's preference list
                    curr_partner = women_matchings[woman]
                    curr_partner_rank = wpref[woman].index(curr_partner)
                    man_rank = wpref[woman].index(man + 1)

                    # woman prefers curr_partner over man
                    if curr_partner_rank < man_rank:
                        last_rejected_by_rank_what[man] = mpref[man].index(woman + 1) + 1

                    # woman prefers man over curr_partner
                    else:
                        men_matchings[curr_partner - 1] = 0
                        last_rejected_by_rank_what[curr_partner - 1] = mpref[curr_partner - 1].index(woman + 1) + 1
                        men_matchings[man] = woman + 1
                        women_matchings[woman] = man + 1

    if strategic_woman != 0:
        return men_matchings, proposals_to_strategic_woman
    else:
        return men_matchings


def get_m_partner(m, matches):
    return matches[m - 1]


def get_w_partner(w, matches):
    return matches.index(w) + 1


def get_m_rank(m, mpref, matches):
    partner = matches[m - 1]
    return mpref[m - 1].index(partner) + 1


def get_w_rank(w, wpref, matches):
    partner = matches.index(w) + 1
    return wpref[w - 1].index(partner) + 1


# Checks if matching1 Pareto-improves matching2 for the set of all women
def pareto_improvement_women(wpref, matching1, matching2):
    n = len(matching1)
    matching1_ranks = [get_w_rank(w, wpref, matching1) for w in range(1, n + 1)]
    matching2_ranks = [get_w_rank(w, wpref, matching2) for w in range(1, n + 1)]
    ranks_difference = [matching2_ranks[i] - matching1_ranks[i] for i in range(n)]
    improvement = False
    for diff in ranks_difference:
        if diff < 0:
            return False
        if diff > 0:
            improvement = True
    return improvement


def check_stability(mpref, wpref, matching, return_blocking_pairs=False):
    n = len(mpref)
    blocking_pairs = []

    for m in range(1, n + 1):
        m_partner = matching[m - 1]
        rank_of_m_partner = mpref[m - 1].index(m_partner) + 1
        for w in range(1, n + 1):
            w_partner = matching.index(w) + 1
            rank_of_w_partner = wpref[w - 1].index(w_partner) + 1

            # if (m, w) is not a matching
            if matching[m - 1] != w:
                rank_of_w_for_m = mpref[m - 1].index(w) + 1
                rank_of_m_for_w = wpref[w - 1].index(m) + 1
                if rank_of_m_partner > rank_of_w_for_m and rank_of_w_partner > rank_of_m_for_w:
                    if return_blocking_pairs:
                        bp = (m, w)
                        blocking_pairs.append(bp)
                    else:
                        return False

    if return_blocking_pairs:
        return blocking_pairs
    else:
        return True



        
    

In [166]:
def invert_proposer_matches(proposer_matches):
    """
    proposer_matches: list of length n where index i (0-based) is proposer (i+1)
                      and value proposer_matches[i] is the partner id (1-based).
    Returns: list `inv` of length n that maps partner -> proposer (1-based ids),
             i.e., inv[partner-1] = proposer_id
    """
    n = len(proposer_matches)
    inv = [0] * n
    for proposer_idx, partner in enumerate(proposer_matches):
        if partner == 0:
            # unmatched / sentinel (shouldn't happen in full matching)
            continue
        inv[partner - 1] = proposer_idx + 1
    return inv


def rank_differences_women(mpref, wpref, debug=False):
    """
    Return list of length n: for each woman w (1..n),
    diff[w-1] = rank_w(men-opt partner for w) - rank_w(women-opt partner for w).
    This should be >= 0 for all w (woman-optimal is at least as good).
    """
    # men-optimal: list mapping man -> woman (1-based IDs)
    men_optimal = gale_shapley(mpref, wpref)

    # women-proposing: returns list mapping woman -> man (1-based IDs)
    women_proposer_matches = gale_shapley(wpref, mpref)

    # invert to get men -> woman mapping for the women-optimal matching
    women_optimal = invert_proposer_matches(women_proposer_matches)

    n = len(mpref)
    differences = []

    for w in range(1, n + 1):
        # partner of woman w in men-optimal matching (this returns the man id who is matched to w)
        partner_men_opt = get_w_partner(w, men_optimal)   # returns man id
        # partner of woman w in women-optimal matching (man id)
        partner_women_opt = get_w_partner(w, women_optimal)

        # rank of those partners in woman's preference list (1-based ranks)
        rank_men_opt = wpref[w - 1].index(partner_men_opt) + 1
        rank_women_opt = wpref[w - 1].index(partner_women_opt) + 1

        diff = rank_men_opt - rank_women_opt
        differences.append(diff)

        if debug and diff < 0:
            print(f"DEBUG NEGATIVE: woman={w}, rank_men_opt={rank_men_opt}, "
                  f"rank_women_opt={rank_women_opt}, man_opt={partner_men_opt}, woman_opt={partner_women_opt}")
            print(" wpref[w-1] =", wpref[w-1])
            print(" men_optimal (man->woman) =", men_optimal)
            print(" women_proposer_matches (woman->man) =", women_proposer_matches)
            print(" women_optimal (man->woman) =", women_optimal)

    return differences


In [170]:
netflix.columns[0]

'Mpref'

In [172]:
import pandas as pd
import ast
import numpy as np

# --- Your helper functions (from before) ---
# gale_shapley, get_w_partner, get_m_partner, rank_differences_women, etc.

def parse_preferences(pref_str):
    """Convert the string representation of preferences to lists."""
    return ast.literal_eval(pref_str)

def compute_differences(pref_str):
    """Compute rank differences for women for a given preference string."""
    mpref, wpref = parse_preferences(pref_str)
    return rank_differences_women(mpref, wpref)

def process_csv(filepath):
    """Load CSV, compute rank differences and average, return updated df."""
    df = pd.read_csv(filepath)
    
    # Identify the preferences column (assuming it's the 3rd column)
    df["preferences"] = df["Mpref"] + "," + df["Wpref"]  # Combine mpref and wpref into a single string
    pref_col = "preferences"  # Assuming the new column contains the preferences
    
    # Compute rank differences
    df["rank_differences"] = df[pref_col].apply(compute_differences)
    
    # Compute average rank difference per row
    df["avg_rank_differences"] = df["rank_differences"].apply(lambda x: np.mean(x))
    
    return df

# --- List of files ---
files = [
    "netflix_preferences.csv",
]

# --- Process all files and save updated CSVs ---
for f in files:
    df_updated = process_csv(f)
    out_file = f.replace(".csv", "_with_rank_diff.csv")
    df_updated.to_csv(out_file, index=False)
    print(f"Saved: {out_file}")


Saved: netflix_preferences_with_rank_diff.csv


In [173]:
df = pd.read_csv("netflix_preferences_with_rank_diff.csv")

# Safely convert string to list (if needed)
def count_positive(lst_str):
    try:
        lst = ast.literal_eval(lst_str)  # convert string -> list
        return sum(1 for x in lst if x > 0)
    except (ValueError, SyntaxError):
        return 0  # fallback if parsing fails

# Create trunc_count column
df["trunc_count"] = df["rank_differences"].apply(count_positive)

df.to_csv("netflix_preferences_with_rank_diff.csv", index=False)

In [4]:
def egalitarian_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n + 1):
        w = matching[m-1]
        cost += mpref[m-1].index(w)
        cost += wpref[w-1].index(m)
    return cost

import numpy as np


def find_all_stable_original(malechoice, femalechoice, n):
    # Output list with all matchings
    res = []
    # Initialization
    unchanged = [True] * n
    marriage = [0] * n
    malecounter = [0] * n
    # Female choice transformation

    fc = np.zeros((n, n), dtype=int)
    for i in range(1, n):
        for j in range(0, n - 1):
            fc[i][femalechoice[i][j]] = j
        fc[i][0] = n
        malecounter[i] = 0
        marriage[i] = 0

    def get_marriage(marriage):
        malematches = [0] * (n - 1)
        for j in range(1, n):
            malematches[marriage[j] - 1] = j
        return(malematches)

    def print_marriage(marriage):
        malematches = [0] * (n - 1)
        for j in range(1, n):
            malematches[marriage[j] - 1] = j
        print(*malematches)

    def proposal(i, malec, marriage, c):
        if i < 0:
            success = True
        elif (i == 0) or malec[i] == (n - 1) or not (unchanged[i]):  # n not (n+1), as we have n equals 5
            success = False
        else:
            c += 1
            j = malec[i]
            malec[i] = j + 1
            success = refusal(i, malechoice[i][j], malec, marriage, c)
        return success

    def refusal(i, j, malec, marriage, c):
        if fc[j][abs(marriage[j])] > fc[j][i]:
            k = marriage[j]
            marriage[j] = i
            success = proposal(k, malec, marriage, c)
        else:
            success = proposal(i, malec, marriage, c)
        return success

    def breakmariage(malecounter, marriage, i, n, count):
        malecounter_ori = malecounter.copy()
        marriage_ori = marriage.copy()
        marriage[malechoice[i][malecounter[i] - 1]] = -i
        success = proposal(i, malecounter, marriage, count)

        if not (success):
            unchanged[i] = False
            malecounter_ref = malecounter_ori.copy()
            marriage_ref = marriage_ori.copy()
            malecounter = malecounter_ref
            marriage = marriage_ref
            return malecounter, marriage

        #print_marriage(marriage)  # marriage[i] means the woman who is married to i man
        new_marriage = get_marriage(marriage)  # convert into female solution
        res.append(new_marriage)
        # establish the relationships between the original and the obtained stable matching

        for j in range(i, n):
            # if i==2 and j==2:
            #     print("This is it")
            malecounter, marriage = breakmariage(malecounter, marriage, j, n, count)

        for j in range((i + 1), n):
            unchanged[j] = True
        # return malecounter_ori, abs(marriage_ori)
        return malecounter_ori, marriage_ori

    count = 0
    for i in range(1, n):
        proposal(i, malecounter, marriage, count)
    #print_marriage(marriage)

    new_marriage = get_marriage(marriage)  # convert into female solution
    res.append(new_marriage)
    # establish the relationships between the original and the obtained stable matching

    M_optimal_marriage = marriage.copy()
    M_optimal_counter = malecounter.copy()
    for i in range(1, n - 1):
        breakmariage(M_optimal_counter.copy(), M_optimal_marriage.copy(), i, n, count)
    return res


In [8]:
df = pd.read_csv("mallows_gain_egal.csv")

egalitarian_matches = []

for idx in range(len(df)):
    row = df.loc[idx]
    mpref, wpref = ast.literal_eval(row["preferences"]) 
    m_pref = [[int(x) for x in sublist] for sublist in mpref]
    w_pref = [[int(x) for x in sublist] for sublist in wpref]
    mpref = [[]] + m_pref
    wpref = [[]] + w_pref

    stable = find_all_stable_original(mpref, wpref, len(mpref))
    best_match = None
    best_cost = float("inf")
    for match in stable:
        cost = egalitarian_cost(match, m_pref, w_pref)
        if cost < best_cost:
            best_cost = cost
            best_match = match

    egalitarian_matches.append(best_match)
df["egalitarian_match"] = egalitarian_matches
df.to_csv("mallows_gain_egal.csv", index=False)

In [9]:
def men_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n+1):
        w = matching[m-1]
        cost += mpref[m-1].index(w)
    return cost

def women_cost(matching, mpref, wpref):
    n = len(matching)
    cost = 0
    for m in range(1, n+1):
        w = matching[m-1]
        cost += wpref[w-1].index(m)
    return cost

In [13]:
df = pd.read_csv("mallows_gain_egal.csv")

men_matches = []

for idx in range(len(df)):
    row = df.loc[idx]
    mpref, wpref = ast.literal_eval(row["preferences"]) 
    m_pref = [[int(x) for x in sublist] for sublist in mpref]
    w_pref = [[int(x) for x in sublist] for sublist in wpref]
    mpref = [[]] + m_pref
    wpref = [[]] + w_pref

    stable = find_all_stable_original(mpref, wpref, len(mpref))
    best_match = None
    best_cost = float("inf")
    for match in stable:
        cost = men_cost(match, m_pref, w_pref)
        if cost < best_cost:
            best_cost = cost
            best_match = match

    men_matches.append(best_match)
df["men_opt_match"] = men_matches
df.to_csv("mallows_gain_egal.csv", index=False)


women_matches = []

for idx in range(len(df)):
    row = df.loc[idx]
    mpref, wpref = ast.literal_eval(row["preferences"]) 
    m_pref = [[int(x) for x in sublist] for sublist in mpref]
    w_pref = [[int(x) for x in sublist] for sublist in wpref]
    mpref = [[]] + m_pref
    wpref = [[]] + w_pref

    stable = find_all_stable_original(mpref, wpref, len(mpref))
    best_match = None
    best_cost = float("inf")
    for match in stable:
        cost = women_cost(match, m_pref, w_pref)
        if cost < best_cost:
            best_cost = cost
            best_match = match

    women_matches.append(best_match)
df["women_opt_match"] = women_matches
df.to_csv("mallows_gain_egal.csv", index=False)


In [14]:
import pandas as pd
df = pd.read_csv("mallows_gain_egal.csv")
import ast
from ast import literal_eval

cost_diffs = []

for idx in range(len(df)):
    row = df.loc[idx]
    mpref, wpref = ast.literal_eval(row["preferences"]) 
    m_pref = [[int(x) for x in sublist] for sublist in mpref]
    w_pref = [[int(x) for x in sublist] for sublist in wpref]
    mpref = [[]] + m_pref
    wpref = [[]] + w_pref
    stable = find_all_stable_original(mpref, wpref, len(mpref))

    egal = ast.literal_eval(row["egalitarian_match"])
    egal_cost = egalitarian_cost(egal, m_pref, w_pref)

    men_opt = ast.literal_eval(row["men_opt_match"])
    men_cost1 = men_cost(men_opt, m_pref, w_pref)


    women_opt = ast.literal_eval(row["women_opt_match"])
    women_cost1 = women_cost(women_opt, m_pref, w_pref)

    delta = egal_cost - men_cost1 - women_cost1
    cost_diffs.append(delta)



df["trunc_egal"] = cost_diffs 
df.to_csv("mallows_gain_egal.csv", index=False)

In [180]:
df = pd.read_csv("netflix_preferences_with_rank_diff.csv")

stability_counts = []

for idx in range(len(df)):
    row = df.loc[idx]
    mpref, wpref = ast.literal_eval(row["preferences"]) 
    m_pref = [[int(x) for x in sublist] for sublist in mpref]
    w_pref = [[int(x) for x in sublist] for sublist in wpref]
    mpref = [[]] + m_pref
    wpref = [[]] + w_pref
    stable = find_all_stable_original(mpref, wpref, len(mpref))
    num_stable = len(stable)
    stability_counts.append(num_stable)


df["stability_count"] = stability_counts
df.to_csv("netflix_preferences_with_rank_diff.csv", index=False)

In [181]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
 
# Load the CSV
df = pd.read_csv("netflix_preferences_with_rank_diff.csv")
 
# Create trunc_prevalence column: 1 if stability_count > 1, else 0
df["trunc_prevalence"] = (df["stability_count"] > 1).astype(int)
 
df.to_csv("netflix_preferences_with_rank_diff.csv", index=False)

**Accomplice Testing**

In [221]:
def gale_shapley(mpref, wpref, strategic_woman=0):
    n = len(mpref)
    curr_match_count = 0
    last_rejected_by_rank_what = [0 for i in range(n)]
    men_matchings = [0 for i in range(n)]
    women_matchings = [0 for i in range(n)]
    proposals_to_strategic_woman = set()    # Only used if strategic_woman != 0

    # continue until all matches made
    while curr_match_count < n:
        for man in range(n):
            # if man is single
            if men_matchings[man] == 0:
                # get first non-rejecting woman from man's preference list
                woman = mpref[man][last_rejected_by_rank_what[man]] - 1
                if woman + 1 == strategic_woman:
                    proposals_to_strategic_woman.add(man + 1)

                # both man and woman are single
                if women_matchings[woman] == 0:
                    # create match (man, woman)
                    men_matchings[man] = woman + 1
                    women_matchings[woman] = man + 1
                    curr_match_count += 1

                # woman is with someone
                else:
                    # find ranks of curr_partner and man given woman's preference list
                    curr_partner = women_matchings[woman]
                    print(type(wpref[woman]))
                    curr_partner_rank = wpref[woman].index(curr_partner)
                    man_rank = wpref[woman].index(man + 1)

                    # woman prefers curr_partner over man
                    if curr_partner_rank < man_rank:
                        last_rejected_by_rank_what[man] = mpref[man].index(woman + 1) + 1

                    # woman prefers man over curr_partner
                    else:
                        men_matchings[curr_partner - 1] = 0
                        last_rejected_by_rank_what[curr_partner - 1] = mpref[curr_partner - 1].index(woman + 1) + 1
                        men_matchings[man] = woman + 1
                        women_matchings[woman] = man + 1

    if strategic_woman != 0:
        return men_matchings, proposals_to_strategic_woman
    else:
        return men_matchings

def get_m_partner(m, matches):
    return matches[m - 1]


def get_w_partner(w, matches):
    return matches.index(w) + 1


def get_m_rank(m, mpref, matches):
    partner = matches[m - 1]
    return mpref[m - 1].index(partner) + 1


def get_w_rank(w, wpref, matches):
    partner = matches.index(w) + 1
    return wpref[w - 1].index(partner) + 1

In [239]:
#instead of putting multiple of the same blocks, im just going to replace the file names

import tqdm
from AccompliceManipulation import *

df = pd.read_csv("netflix_preferences_with_rank_diff.csv")

results = []
original_matchings = [] 

for idx, row in tqdm(df.iterrows()):
    mpref, wpref = ast.literal_eval(row["preferences"]) 
    mpref = [[int(x) for x in sublist] for sublist in mpref]
    wpref = [[int(x) for x in sublist] for sublist in wpref]

    n = len(mpref)

    original_matching = gale_shapley(mpref, wpref)
    original_matchings.append(str(original_matching))

    instance_results = []  # store all manipulator-accomplice outcomes for this row

    for manipulator in range(1, n+1):
        manipulator_results = []  # all accomplices for this manipulator

        for accomplice in range(1, n+1):
            func_return = accomplice_manipulation(mpref, wpref, manipulator, accomplice)
            new_matching = func_return[0]
            accomplice_pref_after = func_return[1]

            # --- compute benefit ---
            rk1 = rk2 = 1000
            for i in range(n):
                if original_matching[i] == manipulator:
                    partner1 = i + 1
                if new_matching[i] == manipulator:
                    partner2 = i + 1
            
            w_pref = wpref[manipulator - 1]
            for i in range(n):
                if w_pref[i] == partner1:
                    rk1 = i
                if w_pref[i] == partner2:
                    rk2 = i

            benefit = rk1 - rk2  # improvement (positive = better)

            # store for this accomplice
            manipulator_results.append([
                benefit,
                new_matching,
                accomplice_pref_after,
            ])

        # store for this manipulator
        instance_results.append(manipulator_results)

    # store results for this row
    results.append(instance_results)

#print(results)

# attach to df
df["accomplice_results"] = results
df.to_csv('netflix_preferences_with_rank_diff.csv', index=False)


150it [00:00, 5736.19it/s]


In [241]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ast  # to convert stringified lists to actual lists

# --- Load your CSV ---
df = pd.read_csv("netflix_preferences_with_rank_diff.csv")  # Uncomment and set filename if needed

# Prepare lists to store the new columns
best_benefits_list = []
avg_best_benefit_list = []
acc_freq_instance_list = []
number_acc_pairs_list = []
number_acc_manipulators_list = []

for idx, row in df.iterrows():
    # Convert string to list if needed
    accomplice_results = row["accomplice_results"]
    if isinstance(accomplice_results, str):
        accomplice_results = ast.literal_eval(accomplice_results)

    # Best benefit per manipulator
    best_benefits_per_woman = [
        max(result[0] for result in manipulator_results)
        for manipulator_results in accomplice_results
    ]
    best_benefits_list.append(best_benefits_per_woman)

    # Average across manipulators
    avg_val = np.mean(best_benefits_per_woman)
    avg_best_benefit_list.append(avg_val)

    # --- New columns ---
    # 1. acc_freq_instance
    acc_freq_instance_list.append(1 if avg_val > 0 else 0)

    # 2. number_acc_pairs: manipulator has accomplice with benefit > 0
    #num_pairs = sum(
        #any(result[0] > 0 for result in manipulator_results)
        #for manipulator_results in accomplice_results
    #)
    #number_acc_pairs_list.append(num_pairs)

    num_pairs = sum(
    sum(1 for result in manipulator_results if result[0] > 0)
    for manipulator_results in accomplice_results
    )
    number_acc_pairs_list.append(num_pairs)

    # 3. number_acc_manipulators: count of >0 best benefits
    num_manipulators = sum(1 for b in best_benefits_per_woman if b > 0)
    number_acc_manipulators_list.append(num_manipulators)

# Add new columns to the original DataFrame
df["best_benefits_per_woman_acc"] = best_benefits_list
df["avg_best_benefit_acc"] = avg_best_benefit_list
df["acc_freq_instance"] = acc_freq_instance_list
df["number_acc_pairs"] = number_acc_pairs_list
df["number_acc_manipulators"] = number_acc_manipulators_list

# --- Save the updated DataFrame to CSV ---
df.to_csv("netflix_preferences_with_rank_diff.csv", index=False)


In [242]:
def self_manipulation(mpref, wpref, manipulator, return_all_strategies=False):
    n = len(mpref)

    # initial run of gale_shapley
    gs_matching, all_proposers = gale_shapley(mpref, wpref, manipulator)
    #print(gale_shapley(mpref, wpref, manipulator))
    exhausted_proposers = [get_w_partner(manipulator, gs_matching)]

    proposer_dict = {}  # key = proposer; value = manipulated  preference list
    strategy = (gs_matching, wpref[manipulator - 1][:])
    all_strategies = [strategy]

    # update proposer_dict to include all proposers from the initial run
    for proposer in all_proposers:
        proposer_dict.update({proposer: wpref[manipulator - 1][:]})

    # loop until all potential proposers are exhausted
    while len(exhausted_proposers) != len(all_proposers):
        all_proposers_copy = all_proposers.copy()

        # for all proposers that haven't been exhausted
        for proposer in all_proposers:
            if proposer not in exhausted_proposers:
                # create a new manipulated list with proposer in front
                new_list = proposer_dict[proposer]
                new_list.remove(proposer)
                new_list.insert(0, proposer)
                exhausted_proposers.append(proposer)

                # update each non-exhausted proposer's list in proposer_dict
                for prop in proposer_dict:
                    if prop not in exhausted_proposers:
                        proposer_dict.update({proposer: new_list[:]})

                # run gale_shapley using the new manipulated list
                wpref_new = wpref[:]
                wpref_new[manipulator - 1] = new_list
                new_matching, new_proposers = gale_shapley(mpref, wpref_new, manipulator)
                all_proposers_copy = all_proposers_copy.union(new_proposers)
                strategy = (new_matching[:], new_list[:])
                all_strategies.append(strategy)

                # update proposer_dict to include any new proposers
                for prop in all_proposers_copy:
                    if prop not in exhausted_proposers and prop not in proposer_dict:
                        proposer_dict.update({prop: wpref_new[manipulator - 1][:]})

        # update all_proposers
        all_proposers = all_proposers_copy

    if return_all_strategies:
        return all_strategies

    # find cheater's best ranking man from all those who proposed
    best_man_rank = n + 1
    best_strategy = None
    for strategy in all_strategies:
        rank = get_w_rank(manipulator, wpref, strategy[0])
        if rank < best_man_rank:
            best_man_rank = rank
            best_strategy = strategy

    return best_strategy


In [260]:
df = pd.read_csv("4_expectations_euclidean_with_rank_diff.csv")

results = []

for idx, row in tqdm(df.iterrows()):

    mpref, wpref = ast.literal_eval(row["preferences"]) 
    mpref = [[int(x) for x in sublist] for sublist in mpref]
    wpref = [[int(x) for x in sublist] for sublist in wpref]
    

    n = len(mpref)

    benefit_women = []
    manipulated_matchings = []
    original_matchings = []
    manipulated_preferences = []

    for manipulator in range (1, n+1):
        matching = gale_shapley(mpref, wpref)
        result = self_manipulation(mpref, wpref, manipulator)
        matching2 = result[0]
        rk1 = 1000
        rk2 = 1000

        for i in range(0,n):
            if matching[i] == manipulator:
                partner1=i+1
            if matching2[i] == manipulator:
                partner2=i+1
        w_pref = wpref[manipulator-1]
        
        for i in range(0,n):
            if w_pref[i] == partner1:
                rk1=i
            if w_pref[i] == partner2:
                rk2=i
        manipulated_matchings.append(matching2)
        original_matchings.append(matching)
        manipulated_preferences.append(result[1])
        benefit = abs(rk1-rk2)
        benefit_women.append(benefit)
    results.append(benefit_women)

df["benefit"] = results
df.to_csv("4_expectations_euclidean_with_rank_diff.csv", index=False)

1500it [00:00, 10295.70it/s]


In [261]:
df = pd.read_csv("4_expectations_euclidean_with_rank_diff.csv")

df["self_benefit_parsed"] = df["benefit"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df["avg_self_benefit"] = df["self_benefit_parsed"].apply(
    lambda lst: np.max(lst) if lst else 0
)

# (b) number of manipulators = number of entries > 0
df["num_self_manipulators"] = df["self_benefit_parsed"].apply(
    lambda lst: np.sum(np.array(lst) > 0) if lst else 0
)

# (c) frequency of instance = 1 if any manipulator exists
df["self_freq_instance"] = df["num_self_manipulators"].apply(
    lambda x: 1 if x > 0 else 0
)

df.to_csv("4_expectations_euclidean_with_rank_diff.csv", index=False)

In [264]:
## Two for One Code

import itertools
import copy

def generate_sw_manipulations(w_pref):
    """
    Sw: Move any two distinct men to top of w's list in any order.
    """
    manipulations = []
    for mi, mj in itertools.permutations(w_pref, 2): 
        rest = [m for m in w_pref if m != mi and m != mj]
        new_pref = [mi, mj] + rest
        manipulations.append(new_pref)
    return manipulations


def generate_sm_manipulations(m_pref, mu_m):
    """
    Sm: Promote µ(m) to top, then for every other woman, promote her to top of that list.
    """
    sm = []
    bm = [mu_m] + [w for w in m_pref if w != mu_m]
    sm.append(bm)

    for w0 in bm[1:]:  # exclude mu(m) which is already at the top
        manipulated = [w0] + [w for w in bm if w != w0]
        sm.append(manipulated)

    return sm


def two_for_one(mpref, wpref, accomplice, beneficiary, initial_matching):
    #initial_matching = gale_shapley(mpref, wpref)
    mpref_prime = copy.deepcopy(mpref)
    wpref_prime = copy.deepcopy(wpref)
    best_matching = initial_matching

    accomplice_pref = mpref[accomplice - 1] 
    beneficiary_pref = wpref[beneficiary - 1]

    Sw = generate_sw_manipulations(beneficiary_pref)

    for w_misreport in Sw:
        #t5 = time.perf_counter()

        Sm = generate_sm_manipulations(accomplice_pref, initial_matching[accomplice - 1])

        for m_misreport in Sm:
            #t6 = time.perf_counter()
            mpref_prime[accomplice - 1] = m_misreport 
            wpref_prime[beneficiary - 1] = w_misreport
            new_matching = gale_shapley(mpref_prime, wpref_prime)

            new_w_pairing = new_matching[beneficiary - 1]
            new_m_pairing = new_matching[accomplice - 1]

            if beneficiary_pref.index(new_w_pairing) < beneficiary_pref.index(best_matching[beneficiary - 1]) and best_matching[accomplice - 1] == new_m_pairing:
                best_matching = new_matching
                accomplice_pref = m_misreport
                beneficiary_pref = w_misreport
        #t7 = time.perf_counter()
        #print(f"Time for t5 - t6: {t6 - t5:.4f} seconds")
        #print(f"Time for t6 - t7: {t7 - t6:.4f} seconds")
    return best_matching, accomplice_pref, beneficiary_pref

In [277]:
df = pd.read_csv("netflix_preferences_with_rank_diff.csv")

results = []
original_matchings = []

for idx, row in tqdm(df.iterrows()):
    mpref, wpref = ast.literal_eval(row["preferences"]) 
    mpref = [[int(x) for x in sublist] for sublist in mpref]
    wpref = [[int(x) for x in sublist] for sublist in wpref]

    n = len(mpref)
    #original_matching = df["original matchings"][idx]

    original_matching = gale_shapley(mpref, wpref)
    original_matchings.append(str(original_matching))

    instance_results = []  # store all manipulator-accomplice outcomes for this row

    for manipulator in range(1, n+1):
        manipulator_results = []  # all accomplices for this manipulator

        for accomplice in range(1, n+1):
            func_return = two_for_one(mpref, wpref, accomplice, manipulator, original_matching)
            new_matching = func_return[0]
            accomplice_pref_after, beneficiary_pref_after = func_return[1:]

            # --- compute benefit ---
            rk1 = rk2 = 1000
            for i in range(n):
                if original_matching[i] == manipulator:
                    partner1 = i + 1
                if new_matching[i] == manipulator:
                    partner2 = i + 1
            
            w_pref = wpref[manipulator - 1]
            for i in range(n):
                if w_pref[i] == partner1:
                    rk1 = i
                if w_pref[i] == partner2:
                    rk2 = i

            benefit = abs(rk1 - rk2)  # improvement (positive = better)

            # store for this accomplice
            manipulator_results.append([
                benefit,
                new_matching,
                accomplice_pref_after,
                beneficiary_pref_after
            ])

        # store for this manipulator
        instance_results.append(manipulator_results)

    # store results for this row
    results.append(instance_results)

#print(results)

# attach to df
df["two_for_one_results"] = results
df.to_csv('netflix_preferences_with_rank_diff.csv', index=False)


150it [00:00, 612.45it/s]


In [285]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ast  # to convert stringified lists to actual lists

# --- Load your CSV ---
df = pd.read_csv("netflix_preferences_with_rank_diff.csv")

# Prepare lists to store the new columns
best_benefits_list = []
avg_best_benefit_list = []
acc_freq_instance_list = []
number_acc_pairs_list = []
number_acc_manipulators_list = []

for idx, row in df.iterrows():
    # Convert string to list if needed
    two_for_one_results = row["two_for_one_results"]
    if isinstance(two_for_one_results, str):
        two_for_one_results = ast.literal_eval(two_for_one_results)

    # Best benefit per manipulator
    best_benefits_per_woman = [
        max(result[0] for result in manipulator_results)
        for manipulator_results in two_for_one_results
    ]
    best_benefits_list.append(best_benefits_per_woman)

    # Average across manipulators
    avg_val = np.mean(best_benefits_per_woman)
    avg_best_benefit_list.append(avg_val)

    # --- New columns ---
    acc_freq_instance_list.append(1 if avg_val > 0 else 0)

    # number_acc_pairs: manipulator has at least one accomplice with benefit > 0
    #num_pairs = sum(
        #any(result[0] > 0 for result in manipulator_results)
        #for manipulator_results in two_for_one_results
    #)
    #number_acc_pairs_list.append(num_pairs)
    
    
    num_pairs = sum(
    sum(1 for result in manipulator_results if result[0] > 0)
    for manipulator_results in two_for_one_results
    )
    number_acc_pairs_list.append(num_pairs)

    # number_acc_manipulators: count of >0 best benefits
    num_manipulators = sum(1 for b in best_benefits_per_woman if b > 0)
    number_acc_manipulators_list.append(num_manipulators)

# Add new columns to the original DataFrame
#df["best_benefits_per_woman_two"] = best_benefits_list
df["avg_best_benefit_two"] = avg_best_benefit_list
#df["two_freq_instance"] = acc_freq_instance_list
#df["number_two_pairs"] = number_acc_pairs_list
#df["number_two_manipulators"] = number_acc_manipulators_list

# --- Save the updated DataFrame to CSV ---
df.to_csv("netflix_preferences_with_rank_diff.csv", index=False)